# GiTrust × Kev-0.8B — round 2 fine-tune (`gitrust-v2`) on Colab, serve locally

Same recipe as round 1 (`kev_finetune_colab.ipynb` → `gitrust-v1`): a LoRA delta on top of the released **Kev-0.8B
v1.0**, from the stock checkpoint, so v1 and v2 differ only in their data. Round 2 adds the `commit_push` option
(36 options now), typo variants, casual history phrasings, "push all <topic> changes", history-wiping
`delete_repo` phrasings and Git-adjacent non-Git requests (see `make_kev_train.py --round 2`).

New in round 2: step 4 **fits the calibration temperature** on the dev split and writes it into `head.pt`
(round 1 came back at T = 1.0, uncalibrated, and had to be fixed on the laptop).

**Runtime → Change runtime type → GPU** (a free T4 is enough). Run top to bottom: ~2 h on a T4 (≈370 optimizer
steps; round 1 took ~100 min for 314).

Upload when prompted: `kev_train_r2.jsonl` and `kev_dev_r2.jsonl` from `Glit/docs/classifier_eval/`.
**Never upload `eval_set.json` or `eval_set_v2.json`** — they are the held-out tests, scored on the laptop.


In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv
import torch; print("bf16 supported:", torch.cuda.is_bf16_supported())

## 1 · Install Kev (pinned to the commit tested locally)

In [ ]:
!curl -LsSf https://astral.sh/uv/install.sh | sh
import os; os.environ["PATH"] = os.path.expanduser("~/.local/bin") + ":" + os.environ["PATH"]
!git clone -q https://github.com/jaredpalmer/kev.git /content/kev
%cd /content/kev
!git checkout -q 85ccf56a0afb952323c621055b02876e8498d60c && git log -1 --oneline
!uv sync --extra serve 2>&1 | tail -3

## 2 · Upload the round-2 training data


In [ ]:
from google.colab import files
up = files.upload()   # pick kev_train_r2.jsonl and kev_dev_r2.jsonl
assert {"kev_train_r2.jsonl", "kev_dev_r2.jsonl"} <= set(up), up.keys()
assert not {"eval_set.json", "eval_set_v2.json"} & set(up), "held-out eval files must not be uploaded"
import json
opts = json.loads(open("kev_train_r2.jsonl").readline())["questions"]["op"]["criteria"]
assert "commit_push" in opts and len(opts) == 36, f"expected the 36-op round-2 data, got {len(opts)} options"
!wc -l kev_train_r2.jsonl kev_dev_r2.jsonl


## 3 · Train

Identical flags to round 1, new data and output folder (`runs/gitrust-v2`).
* `--init_from jaredpalmer/kev-0.8b@v1.0` — delta mode from the released model (not from v1, so the two rounds compare cleanly).
* `--perm_kl 1.0` — penalises answers that change when the options are reordered.
* fp32 unless the GPU does bf16 (a T4 does not); `--checkpointing 1` keeps activations small.


In [ ]:
import torch, subprocess
dtype = "bf16" if torch.cuda.is_bf16_supported() else "fp32"
cmd = ("uv run python -m kev.train --data kev_train_r2.jsonl --base Qwen/Qwen3.5-0.8B-Base "
       "--init_from jaredpalmer/kev-0.8b@v1.0 --epochs 2 --lr 2e-5 --batch 1 --accum 8 "
       f"--checkpointing 1 --dtype {dtype} --device cuda --perm_kl 1.0 --seed 0 --out runs/gitrust-v2")
print(cmd)
!{cmd} 2>&1 | grep -v "it/s" | tail -40
!ls -la runs/gitrust-v2


## 4 · Dev check (stock vs fine-tuned) + calibration

Serves each checkpoint and scores `kev_dev_r2.jsonl` (phrasings never trained on, but from the same templates —
a sanity check, not the real test). For the fine-tune it keeps every row's full probability distribution, fits a
temperature by minimum NLL over Kev's 0.25–4 log grid, writes it into `runs/gitrust-v2/head.pt` (with provenance;
the T=1 original is kept as `head.pt.T1-backup`), then re-serves to confirm the server reports it.


In [ ]:
import json, os, signal, subprocess, time, urllib.request

def models(port):
    return json.loads(urllib.request.urlopen(f"http://127.0.0.1:{port}/v1/models", timeout=2).read())["models"][0]

def port_free(port):
    try:
        models(port); return False
    except Exception:
        return True

def serve(run, port):
    # `uv run` spawns python as a CHILD: killing only `uv` leaves the server on
    # the port, and the next "serve" would silently score the OLD model. So:
    # own process group, kill the group, and verify which run is answering.
    assert port_free(port), f"port {port} already in use — run: !pkill -f kev.serve"
    env = dict(os.environ, KEV_DTYPE="fp32")
    p = subprocess.Popen(["uv", "run", "--extra", "serve", "python", "-m", "kev.serve", "--run", run, "--port", str(port)],
                         env=env, stdout=open(f"serve-{port}.log", "w"), stderr=subprocess.STDOUT, start_new_session=True)
    for _ in range(180):
        try:
            m = models(port)
            assert m["run"].rstrip("/").endswith(run.rstrip("/")), f"wrong model answering: {m['run']} (expected {run})"
            print("serving", m["run"]); return p
        except AssertionError:
            stop(p); raise
        except Exception:
            time.sleep(2)
    stop(p); raise RuntimeError(open(f"serve-{port}.log").read()[-2000:])

def stop(p):
    try: os.killpg(p.pid, signal.SIGTERM)
    except ProcessLookupError: pass
    p.wait()
    for _ in range(30):
        if port_free(8009): return
        time.sleep(1)
    raise RuntimeError("server did not stop")

def score(port):
    """Returns (correct, n, rows) — rows keep the full distribution for the temperature fit."""
    ok = n = 0; rows = []
    for line in open("kev_dev_r2.jsonl"):
        r = json.loads(line); q = r["questions"]["op"]; gold = q.pop("label")
        body = {"model": "kev-latest", "state": r["state"], "questions": {"op": q}}
        req = urllib.request.Request(f"http://127.0.0.1:{port}/v1/systemone", data=json.dumps(body).encode(),
                                     headers={"content-type": "application/json"})
        a = json.loads(urllib.request.urlopen(req, timeout=120).read())["answers"]["op"]
        ok += a["choice"] == gold; n += 1
        rows.append({"gold": gold, "probs": a["probabilities"]})
    return ok, n, rows

results = {}
for name, run in [("stock", "jaredpalmer/kev-0.8b@v1.0"), ("fine-tuned", "runs/gitrust-v2")]:
    p = serve(run, 8009)
    try:
        served_T = models(8009)["temperature"]
        ok, n, rows = score(8009); results[name] = (served_T, rows)
        print(f"{name:11} dev accuracy {ok}/{n} = {100*ok/n:.1f}%   (served at T={served_T:.3f})")
    finally:
        stop(p)


In [ ]:
# Fit the temperature on the fine-tune's dev distributions (only valid if they were served at T=1).
import math, shutil, datetime
served_T, rows = results["fine-tuned"]
assert abs(served_T - 1.0) < 1e-6, f"fine-tune served at T={served_T}, not 1.0 — the fit would be relative; stop here"

def tempered(probs, T):
    logs = {k: math.log(max(v, 1e-12)) / T for k, v in probs.items()}
    m = max(logs.values()); z = sum(math.exp(v - m) for v in logs.values())
    return {k: math.exp(v - m) / z for k, v in logs.items()}
def nll(T): return -sum(math.log(max(tempered(r["probs"], T)[r["gold"]], 1e-12)) for r in rows) / len(rows)
def ece(T, bins=10):
    b = [[0, 0.0, 0] for _ in range(bins)]
    for r in rows:
        p = tempered(r["probs"], T); top = max(p, key=p.get); i = min(int(p[top] * bins), bins - 1)
        b[i][0] += 1; b[i][1] += p[top]; b[i][2] += top == r["gold"]
    return sum(abs(c - k) for n, c, k in b if n) / len(rows)

grid = [0.25 * (16 ** (i / 120)) for i in range(121)]
T = min(grid, key=nll)
print(f"fitted T={T:.3f}: NLL {nll(1.0):.3f} -> {nll(T):.3f}, ECE {ece(1.0):.3f} -> {ece(T):.3f}")

head = "runs/gitrust-v2/head.pt"
shutil.copy(head, head + ".T1-backup")
h = torch.load(head, map_location="cpu", weights_only=False)
h["temperature"] = round(T, 3)
h["temperature_fit"] = {
    "method": "min mean NLL over a 121-point log grid 0.25..4 (GiTrust kev_finetune_colab_r2.ipynb step 4)",
    "rows": f"GiTrust kev_dev_r2.jsonl: {len(rows)} held-out phrasings", "in_distribution": True,
    "note": "dev phrasings share templates with training; fitted from served T=1 probabilities",
    "nll": {"T=1": round(nll(1.0), 3), f"T={T:.3f}": round(nll(T), 3)},
    "ece": {"T=1": round(ece(1.0), 3), f"T={T:.3f}": round(ece(T), 3)},
    "date": datetime.date.today().isoformat()}
torch.save(h, head)

p = serve("runs/gitrust-v2", 8009)
try:
    print("server now reports temperature", models(8009)["temperature"])
finally:
    stop(p)


## 5 · Download the run folder

Unzip it into `~/Desktop/personal/kev/runs/` on the laptop (next to `gitrust-v1`).


In [ ]:
!cd runs && zip -qr /content/gitrust-v2.zip gitrust-v2 && ls -la /content/gitrust-v2.zip
files.download("/content/gitrust-v2.zip")
